In [3]:
!pip install -U evaluate -q

In [4]:
import json
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
import numpy as np
import evaluate

# Paths: adjust to your dataset mount
DATA_DIR = "/kaggle/input/kharagpur-data-science-hackathon-kdsh-2026-dataset" 
TRAIN_PATH = f"{DATA_DIR}/train.csv"
TEST_PATH  = f"{DATA_DIR}/test.csv"

MC_CONS_PATH  = "/kaggle/input/kdsh26-jsonl-file-characters/monte_cristo/monte_cristo_constraints_updated.jsonl"
CA_CONS_PATH  = "/kaggle/input/kdsh26-jsonl-file-characters/castaways/castaways_constraints_filled.jsonl"

train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)
print(train_df.head())
print(train_df["label"].value_counts())


2026-01-10 03:14:18.141989: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768014858.467163      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768014858.562304      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768014859.312494      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768014859.312551      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768014859.312556      55 computation_placer.cc:177] computation placer alr

    id                   book_name        char  \
0   46  In Search of the Castaways    Thalcave   
1  137   The Count of Monte Cristo       Faria   
2   74  In Search of the Castaways  Kai-Koumou   
3  109   The Count of Monte Cristo    Noirtier   
4  104   The Count of Monte Cristo    Noirtier   

                                             caption  \
0                                                NaN   
1  The Origin of His Connection with the Count of...   
2                                                NaN   
3         The Complexity of Family and Personal Life   
4  Involvement and Turning Point in the French Re...   

                                             content       label  
0  Thalcave’s people faded as colonists advanced;...  consistent  
1  Suspected again in 1815, he was re-arrested an...  contradict  
2  Before each fight he studied the crack-pattern...  consistent  
3  Villefort’s drift toward the royalists disappo...  contradict  
4  His parents were targete

In [5]:
def load_constraints(path):
    mapping = {}
    with Path(path).open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            key = (obj["book_name"], obj["character"])
            mapping[key] = obj.get("constraints", [])
    return mapping

mc_constraints = load_constraints(MC_CONS_PATH)  # Monte Cristo [code_file:262]
ca_constraints = load_constraints(CA_CONS_PATH)  # Castaways    [code_file:293]
constraints = {**mc_constraints, **ca_constraints}

def constraint_to_sentence(book, char, c):
    dim = c["dimension"]
    val = c["value"]
    if dim == "health_state":
        return f"In {book}, {char} is {val}."
    elif dim == "family_role":
        return f"In {book}, {char} has family role: {val}."
    elif dim == "role":
        return f"In {book}, {char} is described as {val}."
    elif dim == "geographic_expertise":
        return f"{char} is familiar with {val}."
    elif dim == "criminal_history":
        return f"{char} has criminal history: {val}."
    else:
        return f"{dim}: {val}."

def build_context(book, char, max_cons=6):
    cons = constraints.get((book, char), [])
    if not cons:
        return ""
    sents = [constraint_to_sentence(book, char, c) for c in cons[:max_cons]]
    return " ".join(sents)

train_df["context"] = train_df.apply(
    lambda r: build_context(r["book_name"], r["char"]), axis=1
)
test_df["context"] = test_df.apply(
    lambda r: build_context(r["book_name"], r["char"]), axis=1
)

# Quick check
print(train_df[["book_name","char","context","content","label"]].head())


                    book_name        char  \
0  In Search of the Castaways    Thalcave   
1   The Count of Monte Cristo       Faria   
2  In Search of the Castaways  Kai-Koumou   
3   The Count of Monte Cristo    Noirtier   
4   The Count of Monte Cristo    Noirtier   

                                             context  \
0  In In Search of the Castaways, Thalcave is des...   
1                                                      
2  In In Search of the Castaways, Kai-Koumou is d...   
3                                                      
4                                                      

                                             content       label  
0  Thalcave’s people faded as colonists advanced;...  consistent  
1  Suspected again in 1815, he was re-arrested an...  contradict  
2  Before each fight he studied the crack-pattern...  consistent  
3  Villefort’s drift toward the royalists disappo...  contradict  
4  His parents were targeted in a reprisal for su...  con

In [6]:
MODEL_NAME = "microsoft/deberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

label2id = {"consistent": 0, "contradict": 1}
id2label = {v: k for k, v in label2id.items()}

def encode_examples(df, is_train=True):
    texts = [
        f"Premise: {ctx}\nHypothesis: {claim}"
        for ctx, claim in zip(df["context"].fillna(""), df["content"])
    ]
    enc = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=512,
    )
    if is_train:
        enc["labels"] = [label2id[l] for l in df["label"]]
    return enc

# Train/val split (stratified)
train_split, val_split = train_test_split(
    train_df,
    test_size=0.2,
    stratify=train_df["label"],
    random_state=42,
)

train_enc = encode_examples(train_split, is_train=True)
val_enc   = encode_examples(val_split,   is_train=True)
test_enc  = encode_examples(test_df,    is_train=False)

train_ds = Dataset.from_dict(train_enc)
val_ds   = Dataset.from_dict(val_enc)
test_ds  = Dataset.from_dict(test_enc)


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [7]:
import os
os.environ["WANDB_DISABLED"] = "true"

In [8]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1.compute(predictions=preds, references=labels, average="macro")["f1"],
    }

training_args = TrainingArguments(
    output_dir="./deberta-v3-base-nli",
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=8,          # small data, more epochs OK
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=10,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
/tmp/ipykernel_55/3793211843.py:33: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()

SAVE_DIR = "./kaggle/working/deberta-v3-base-castaways-monte" 

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print("Saved fine-tuned model + tokenizer to", SAVE_DIR)


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss


In [ ]:
label_map_path = os.path.join(SAVE_DIR, "label_map.json")
with open(label_map_path, "w") as f:
    json.dump({"label2id": label2id, "id2label": id2label}, f)